# Prepare the features

- [X] BOP features arrangement and averagings
- [O] Library Features arrangement ad averaging
  average per polihedra.
  average per structure.
  atoms in each polyhedra.
- [X] Matminer features cleanup (decide min, max or avg)
- clear feature list from correlations
- outlier detection based on features
- Feature Distribution plots

# Included averages

- moments of the total density of states :  

$$ m_0^i = \left( \sum_{j} m_j^i \right)/N_{atoms} $$

- CP averaged moments, norm to total number of atoms:

$$ 
   m_{CP}^i =  \left( \sum_{j \in CP} m_j^i \right)/N_{atoms}
$$ 

- CP averaged moments, norm atoms in CP:

$$ 
   m_{CP}^i =  \left( \sum_{j \in CP} m_j^i \right)/N_{CP}
$$ 

# but I should also include:

- [X] Number of atoms in each CP

- correlation matrices for pair of sets

 - input : features dataframes from prev steps
 - output: clean features dataframes into picles

In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rc('figure', figsize = (15,10))
plt.rc('font', size=22)
plt.rc('text', usetex=True)
import copy
import os
import numpy as np
import pdb
from tqdm.auto import tqdm 
from Tools.DatasetTools import GeneralFeaturizer as gf
dataset = 'Cr-Co-W'
descriptorlocation = os.path.join(dataset, 'Descriptors')
system = dataset.replace('-','')

import pdb

# Load Features 

In [4]:
AtomicFeaturesFile = os.path.join(descriptorlocation, 'matminer_atomic_features.pkl')
CompositionFeaturesFile = os.path.join(descriptorlocation, 'matminer_composition_features.pkl')

Coordination Number averages

In [5]:
PyscalFeaturesFile = os.path.join(descriptorlocation, 'CNAV_pyscal_steinhardt.kpl')

In [13]:
bopmodels = ['canonical', 'projections', 'projections_os']
bopfeaturesfile = {model: os.path.join(descriptorlocation, f'CNAV_{system}_initial_{model}_table_WUBIND_16.pkl') for model in bopmodels}

In [14]:
PyscalFeatures = pd.read_pickle(PyscalFeaturesFile)
BopFeatures = {model: pd.read_pickle(modelfile) for model, modelfile in bopfeaturesfile.items()}
AtomicFeatures = pd.read_pickle(AtomicFeaturesFile)
CompositionFeatures = pd.read_pickle(CompositionFeaturesFile)

# compiling FULL features 

In [ ]:
AllFeatures = [AtomicFeatures,CompositionFeatures,  BOPAveraged] # 

In [ ]:
FullSetOfFeatures = pd.concat(AllFeatures, axis = 1)

# categorical features transformation

In [ ]:
from sklearn import set_config
from sklearn.compose import make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline

In [ ]:
cat_selector = make_column_selector(dtype_include=object)
categorical_features_names = cat_selector(FullSetOfFeatures)
CategoricalFeatures = FullSetOfFeatures[categorical_features_names]

In [ ]:
oencoder = OrdinalEncoder()
oencoder.fit(X=CategoricalFeatures)
replace_categorical_features_values  = oencoder.transform(CategoricalFeatures)

In [ ]:
FullSetOfFeatures[categorical_features_names] = replace_categorical_features_values

In [ ]:
FullSetOfFeatures[categorical_features_names]

# Fix Column Names  

In [ ]:
def fix_column_name(thecolumn):
    return thecolumn.replace(' ','_')

In [ ]:
columns = FullSetOfFeatures.columns

In [ ]:
newcolumns = {col: fix_column_name(col) for col in columns}

In [ ]:
FullSetOfFeatures.rename(columns=newcolumns, inplace=True)

In [ ]:
FullSetOfFeatures

# Unsupervised feature selection 

## Variance Threashold

remove features with low variance

TODO: for each separate set of features and save clean pickles

In [ ]:
from sklearn.feature_selection import VarianceThreshold

In [ ]:
selector = VarianceThreshold()

In [ ]:
variance_reduced = selector.fit_transform(FullSetOfFeatures)

In [ ]:
selectedfeatures = selector.get_support()

In [ ]:
AfterVarianceFeatures = FullSetOfFeatures.iloc[:,selectedfeatures]

In [ ]:
AfterVarianceFeaturesLocation = os.path.join(descriptorlocation, 'FullSetOfFeatures.pkl')

In [ ]:
FullSetOfFeatures.to_pickle(AfterVarianceFeaturesLocation)

In [ ]:
descriptorlocation

In [ ]:
AfterVarianceFeatures.index.difference(FullSetOfFeatures.index)

In [ ]:
FullSetOfFeatures.index.difference(AfterVarianceFeatures.index)

#  Correlation based feature selection

this is not started even. 

In [ ]:
CORR = AfterVarianceFeatures.corr().abs()

to facilitate higly correlations removal, I will see only the upper triangle of this simmetric matrix

In [ ]:
columns = np.full((CORR.shape[0],), True, dtype=bool)

In [ ]:
tri_upper_corr = CORR.where(np.triu(np.ones(CORR.shape), k=1).astype(bool))

In [ ]:
to_drop = [column for column in tri_upper_corr.columns if any(tri_upper_corr[column] > 0.98)]

In [ ]:
len(to_drop)

In [ ]:
plt.imshow(tri_upper_corr[tri_upper_corr>0.9])
plt.colorbar()

In [ ]:
plt.imshow(CORR)
plt.colorbar()